# 06: Agreement (Cohen's kappa)
Agreement between the independent coder and Opus 5.5 on the same records, reported for the planned 10% subset and for everything coded. Bootstrap 95% CIs resample records. Sparse secondary fields get confusion matrices and raw agreement; do not over-read their kappa.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd, numpy as np
pd.set_option("display.max_colwidth", 80)

In [2]:
from io_utils import load_coder_sheet, load_opus_codes, TABLES
from coding_schema import is_coded, validation_errors, blocking
coder = load_coder_sheet()
coder = coder[coder.apply(is_coded, axis=1)].copy()
assert len(coder), "nothing coded yet"
assert not any(blocking(validation_errors(r)) for _, r in coder.iterrows()), "run 03 and fix validation errors first"
opus = load_opus_codes()
both = coder.merge(opus, on=["arm", "pmid"], suffixes=("_ind", "_opus"))
assert len(both) == len(coder)
print(len(both), "records coded by both;", both.priority_10pct.eq("1").sum(), "of them in the planned 10% subset")

747 records coded by both; 75 of them in the planned 10% subset


In [3]:
from stats_utils import kappa_summary, landis_koch
from io_utils import settings
from coding_schema import DETAIL
s = settings(); reps, seed = s["bootstrap_reps"], s["bootstrap_seed"]
def T_all(df, who):  # Arm B not-relevant becomes its own category so all records enter one kappa
    t = df[f"trajectory_code_{who}"].where(df[f"screen_relevant_{who}"] != "no", "not relevant")
    return t
subsets = {"planned 10% subset": both[both.priority_10pct == "1"], "all coded": both}
res = []
for sname, df in subsets.items():
    A, B = df[df.arm == "A"], df[df.arm == "B"]
    rb = B[(B.screen_relevant_ind == "yes") & (B.screen_relevant_opus == "yes")]
    tests = {
        "T, both arms (Arm B not-relevant as a category)": (T_all(df, "ind"), T_all(df, "opus")),
        "T, Arm A": (A.trajectory_code_ind, A.trajectory_code_opus),
        "Arm B relevance screen": (B.screen_relevant_ind, B.screen_relevant_opus),
        "T, Arm B relevant by both": (rb.trajectory_code_ind, rb.trajectory_code_opus),
        "T collapsed D vs not-D, both arms": (T_all(df, "ind").eq("D"), T_all(df, "opus").eq("D")),
    }
    same_detail = df[(df.trajectory_code_ind == df.trajectory_code_opus) & df.trajectory_code_ind.isin(DETAIL)]
    for f in ["scale", "observation", "anchor", "level"]:
        filled = same_detail[(same_detail[f"{f}_ind"] != "") & (same_detail[f"{f}_opus"] != "")]
        tests[f"{f}, NC/D records with same T"] = (filled[f"{f}_ind"], filled[f"{f}_opus"])
    shaped = df[(df.trajectory_code_ind == df.trajectory_code_opus) & df.trajectory_code_ind.isin(["P", "NC", "D"])]
    shaped = shaped[(shaped.explicitness_ind != "") & (shaped.explicitness_opus != "")]
    tests["explicitness, P/NC/D records with same T"] = (shaped.explicitness_ind, shaped.explicitness_opus)
    tests["organism"] = (df.organism_ind, df.organism_opus)
    for name, (x, y) in tests.items():
        r = kappa_summary(x.astype(str).values, y.astype(str).values, reps=reps, seed=seed)
        res.append({"subset": sname, "comparison": name} | r | {"interpretation": landis_koch(r["cohen_kappa"])})
kappa = pd.DataFrame(res)
kappa.to_csv(TABLES / "kappa_independent_vs_opus.tsv", sep="\t", index=False)
kappa.round(3)

,subset,comparison,n,percent_agreement,cohen_kappa,kappa_ci_low,kappa_ci_high,interpretation
0,planned 10% subset,"T, both arms (Arm B not-relevant as a category)",75,0.893,0.856,0.759,0.943,almost perfect
1,planned 10% subset,"T, Arm A",35,0.914,0.866,0.698,1.000,almost perfect
2,planned 10% subset,Arm B relevance screen,40,0.925,0.808,0.565,1.000,almost perfect
3,planned 10% subset,"T, Arm B relevant by both",9,0.778,0.550,NaN,NaN,moderate
4,planned 10% subset,"T collapsed D vs not-D, both arms",75,1.000,NaN,NaN,NaN,
5,planned 10% subset,"scale, NC/D records with same T",5,1.000,1.000,NaN,NaN,almost perfect
6,planned 10% subset,"observation, NC/D records with same T",5,1.000,NaN,NaN,NaN,
7,planned 10% subset,"anchor, NC/D records with same T",5,1.000,1.000,NaN,NaN,almost perfect
8,planned 10% subset,"level, NC/D records with same T",5,0.800,0.444,NaN,NaN,moderate
9,planned 10% subset,"explicitness, P/NC/D records with same T",16,0.875,0.738,NaN,NaN,substantial


In [4]:
for sname, df in subsets.items():
    cm = pd.crosstab(T_all(df, "opus").rename("Opus 5.5"), T_all(df, "ind").rename("independent"), margins=True)
    cm.to_csv(TABLES / f"confusion_T_{sname.replace(' ', '_').replace('%', 'pct')}.tsv", sep="\t")
    print(sname); display(cm)

planned 10% subset


independent,DEV,NC,P,U,not relevant,All
Opus 5.5,,,,,,
DEV,7,0,0,0,0,7
NC,0,5,1,0,0,6
P,0,1,11,0,0,12
U,0,0,3,16,0,19
not relevant,0,1,2,0,28,31
All,7,7,17,16,28,75


all coded


independent,D,DEV,NC,P,U,not relevant,All
Opus 5.5,,,,,,,
D,7,0,2,0,3,7,19
DEV,0,41,0,0,0,0,41
NC,1,0,25,9,0,27,62
P,1,0,5,86,3,40,135
U,3,5,7,51,135,0,201
not relevant,1,0,4,6,0,278,289
All,13,46,43,152,141,352,747


## Disagreements
Every record where T differs, with both coders' notes, for adjudication. Opus notes are shown here only, after coding is finished.

In [5]:
dis = both[T_all(both, "ind") != T_all(both, "opus")].assign(T_ind=T_all(both, "ind"), T_opus=T_all(both, "opus"))
dis = dis[["review_id", "pmid", "arm", "year", "title", "T_opus", "T_ind", "note", "notes", "evidence_quote"]]
dis = dis.rename(columns={"note": "opus_note", "notes": "independent_notes"})
dis.to_csv(TABLES / "disagreements_T.tsv", sep="\t", index=False)
print(len(dis), "disagreements on T"); dis.head(40)

175 disagreements on T


,review_id,pmid,arm,year,title,T_opus,T_ind,opus_note,independent_notes,evidence_quote
9,A153,20041966,A,2010,Disorders related with ageing in the gerbil female prostate (Skene's paraure...,U,P,prostate lesions in aged female gerbils,,lesions were intensified during senescence
18,A091,9733421,A,1998,Strategies for well-being in later life: a qualitative analysis.,U,P,qualitative well-being case study,,physical slow-down
23,A174,17336301,A,2007,Effects of aging on triggering receptor expressed on myeloid cells (TREM)-1-...,U,P,elderly vs young neutrophil TREM-1 response,,phosphorylation of TREM-1 signal transduction molecules was altered with aging
39,A023,1849052,A,1991,Age-related susceptibility to the carcinogenic effect of the peroxisome prol...,P,U,greater accumulation of initiated hepatocytes with age,,
41,A186,18556430,A,2008,"Physical activity in aging: comparison among young, aged, and nonagenarian i...",P,NC,"physical activity declines with age across young, aged, nonagenarians",,PA was markedly lower in nonagenarians
43,A120,12166367,A,2002,Interactions between texture and trigeminal stimulus in a liquid food system...,U,P,young vs elderly food preference,,preference decreased as the thickness of soups increased
62,A050,8084545,A,1994,Insulin-like growth factor-I receptor densities in human frontal cortex and ...,U,DEV,"receptor density higher in neonates, no decline in adult life",,higher in neonates than in adults
69,A072,8636360,A,1996,Age effects of follicle-stimulating hormone and pulsatile luteinizing hormon...,U,P,gonadotropins in women 40-50 vs 19-39,,age-related increase in FSH concentrations
74,A076,10362857,A,1999,Anaerobic power of the arms and legs of young and older men.,U,P,young vs older anaerobic power,,significant loss of anaerobic power in the older group
77,A288,28882473,A,2017,Age-related changes in sexual function and steroid-hormone receptors in the ...,U,P,mature vs aged male rats sexual function,,As males age they show impairments in sexual function
